# Lab 10 — auditoría, rendimiento y SQL vs pandas

Notebook de las Actividades 5, 9 y 10 sobre `clinical.duckdb`. Incluye la auditoría de calidad, los dos enfoques de rendimiento y la comparación de tres consultas clínicas en DuckDB y pandas.

In [ ]:
from pathlib import Path
from time import perf_counter
import math

import duckdb
import pandas as pd

candidates = [
    Path("../data/clinical.duckdb"),
    Path("lab10/data/clinical.duckdb"),
]
database_path = next((path for path in candidates if path.exists()), None)
if database_path is None:
    raise FileNotFoundError("Ejecuta primero: python3 lab10/build_db.py")

connection = duckdb.connect(str(database_path), read_only=True)
database_path

## Actividad 5 — Auditoría de calidad

Se ejecutan las cuatro comprobaciones directamente sobre DuckDB.

In [ ]:
audit_sql = {
    "IDs duplicados en patients.id": """
        SELECT COUNT(*)
        FROM (
            SELECT id FROM patients GROUP BY id HAVING COUNT(*) > 1
        ) AS duplicated_ids
    """,
    "Encuentros anteriores al nacimiento": """
        SELECT COUNT(*)
        FROM encounters AS e
        JOIN patients AS p ON e.patient_id = p.id
        WHERE CAST(e.start_at AS DATE) < p.birth_date
    """,
    "observations.value nulo o vacío": """
        SELECT
            COUNT(*) AS total,
            COUNT(*) FILTER (
                WHERE value IS NULL OR TRIM(value) = ''
            ) AS missing,
            ROUND(
                100.0 * COUNT(*) FILTER (
                    WHERE value IS NULL OR TRIM(value) = ''
                ) / NULLIF(COUNT(*), 0),
                4
            ) AS missing_percentage
        FROM observations
    """,
    "Encuentros sin paciente": """
        SELECT COUNT(*)
        FROM encounters AS e
        LEFT JOIN patients AS p ON e.patient_id = p.id
        WHERE p.id IS NULL
    """,
}

audit_results = {
    name: connection.execute(query).fetchone() for name, query in audit_sql.items()
}

assert audit_results["IDs duplicados en patients.id"] == (0,)
assert audit_results["Encuentros anteriores al nacimiento"] == (0,)
assert audit_results["observations.value nulo o vacío"] == (17340070, 0, 0.0)
assert audit_results["Encuentros sin paciente"] == (0,)
audit_results

### Interpretación de la auditoría

No se encontraron IDs duplicados, encuentros temporalmente imposibles ni encuentros huérfanos. `observations.value` tuvo 0 valores nulos o vacíos de 17,340,070 (0.0000%). Estos resultados apoyan el uso de `patients.id` como clave lógica y muestran completitud del valor observado en esta carga.

## Actividad 9 — Medición de rendimiento

Se comparan dos implementaciones de pacientes con diabetes tipo 2 (`44054006`) sin HbA1c (`4548-4`). Las consultas se ejecutan para verificar el resultado, pero los tiempos oficiales no se vuelven a medir ni se reemplazan.

In [ ]:
original_not_exists = """
WITH diabetes_patients AS (
    SELECT DISTINCT patient_id
    FROM conditions
    WHERE code = '44054006'
)
SELECT COUNT(*)
FROM diabetes_patients AS d
WHERE NOT EXISTS (
    SELECT 1
    FROM observations AS o
    WHERE o.patient_id = d.patient_id
      AND o.code = '4548-4'
)
"""

rewritten_left_join = """
WITH diabetes_patients AS (
    SELECT DISTINCT patient_id
    FROM conditions
    WHERE code = '44054006'
), hba1c_patients AS (
    SELECT DISTINCT patient_id
    FROM observations
    WHERE code = '4548-4'
)
SELECT COUNT(*)
FROM diabetes_patients AS d
LEFT JOIN hba1c_patients AS h ON d.patient_id = h.patient_id
WHERE h.patient_id IS NULL
"""

performance_result_original = connection.execute(original_not_exists).fetchone()[0]
performance_result_rewritten = connection.execute(rewritten_left_join).fetchone()[0]
assert performance_result_original == performance_result_rewritten == 0

official_performance = pd.DataFrame(
    [
        {"versión": "NOT EXISTS", "resultado": 0, "tiempo oficial (s)": 0.0245},
        {"versión": "CTEs + LEFT JOIN", "resultado": 0, "tiempo oficial (s)": 0.0202},
    ]
)
official_performance

### Interpretación del rendimiento

La reescritura fue aproximadamente 17.6% más rápida en la ejecución oficial (0.0202 s frente a 0.0245 s), pero esto **no** demuestra una mejora estable: son tiempos muy pequeños de una sola ejecución. DuckDB convirtió `NOT EXISTS` en un `RIGHT_ANTI` o anti-join y aplicó *predicate pushdown*, llevando los filtros de código a los escaneos de `conditions` y `observations`. Por ello ambas estrategias fueron eficientes y devolvieron 0 pacientes.

Planes guardados: [`explain_before.txt`](../docs/explain_before.txt) y [`explain_after.txt`](../docs/explain_after.txt).

## Actividad 10 — SQL vs pandas

Los DataFrames se cargan **antes** de iniciar los cronómetros; los tiempos comparan únicamente la operación analítica y no la lectura inicial desde DuckDB o CSV.

### Carga previa de columnas necesarias

Esta celda queda fuera de las mediciones. No se carga `observations`, porque ninguna de las tres comparaciones la necesita.

In [ ]:
patients_df = connection.execute("SELECT id, death_date FROM patients").df()
encounters_df = connection.execute(
    "SELECT id, patient_id, encounter_class FROM encounters"
).df()
conditions_df = connection.execute("SELECT encounter_id FROM conditions").df()

{
    "patients": patients_df.shape,
    "encounters": encounters_df.shape,
    "conditions": conditions_df.shape,
}

## Consulta #1 — pacientes totales y vivos

In [ ]:
sql_1 = """
SELECT
    COUNT(*) AS total_patients,
    COUNT(*) FILTER (WHERE death_date IS NULL) AS alive_patients
FROM patients
"""

start = perf_counter()
sql_result_1 = connection.execute(sql_1).fetchone()
measured_sql_time_1 = perf_counter() - start

start = perf_counter()
pandas_result_1 = (
    len(patients_df),
    int(patients_df["death_date"].isna().sum()),
)
measured_pandas_time_1 = perf_counter() - start

assert sql_result_1 == pandas_result_1 == (22851, 20000)
sql_result_1, pandas_result_1, measured_sql_time_1, measured_pandas_time_1

## Consulta #6 — media y mediana de encuentros por paciente

In [ ]:
sql_6 = """
WITH encounters_per_patient AS (
    SELECT p.id, COUNT(e.id) AS encounter_count
    FROM patients AS p
    LEFT JOIN encounters AS e ON p.id = e.patient_id
    GROUP BY p.id
)
SELECT AVG(encounter_count), MEDIAN(encounter_count)
FROM encounters_per_patient
"""

start = perf_counter()
sql_result_6 = connection.execute(sql_6).fetchone()
measured_sql_time_6 = perf_counter() - start

start = perf_counter()
encounters_per_patient = (
    encounters_df.groupby("patient_id")["id"]
    .count()
    .reindex(patients_df["id"], fill_value=0)
)
pandas_result_6 = (
    float(encounters_per_patient.mean()),
    float(encounters_per_patient.median()),
)
measured_pandas_time_6 = perf_counter() - start

assert math.isclose(sql_result_6[0], pandas_result_6[0], abs_tol=1e-12)
assert sql_result_6[1] == pandas_result_6[1] == 36.0
assert math.isclose(sql_result_6[0], 59.22327250448558, abs_tol=1e-12)
sql_result_6, pandas_result_6, measured_sql_time_6, measured_pandas_time_6

## Consulta #15 — encuentros de urgencias sin diagnóstico

In [ ]:
sql_15 = """
SELECT COUNT(DISTINCT e.id)
FROM encounters AS e
LEFT JOIN conditions AS c ON e.id = c.encounter_id
WHERE e.encounter_class = 'emergency'
  AND c.encounter_id IS NULL
"""

start = perf_counter()
sql_result_15 = connection.execute(sql_15).fetchone()[0]
measured_sql_time_15 = perf_counter() - start

start = perf_counter()
diagnosed_encounter_ids = pd.Index(conditions_df["encounter_id"].dropna().unique())
is_emergency = encounters_df["encounter_class"].eq("emergency")
has_no_diagnosis = ~encounters_df["id"].isin(diagnosed_encounter_ids)
pandas_result_15 = int((is_emergency & has_no_diagnosis).sum())
measured_pandas_time_15 = perf_counter() - start

assert sql_result_15 == pandas_result_15 == 17121
sql_result_15, pandas_result_15, measured_sql_time_15, measured_pandas_time_15

## Comparación oficial

Los tiempos siguientes son los resultados oficiales de la ejecución manual. No se sustituyen por las mediciones variables de esta nueva sesión. Las líneas son aproximadas y excluyen asignaciones, cronómetros y verificaciones.

In [ ]:
comparison = pd.DataFrame(
    [
        {
            "consulta": "#1 Pacientes totales/vivos",
            "resultado": "(22851, 20000)",
            "líneas SQL aprox.": 5,
            "líneas pandas aprox.": 4,
            "tiempo SQL (s)": 0.001432,
            "tiempo pandas (s)": 0.003601,
            "legibilidad": "Ambas son directas; SQL expresa los conteos en una consulta.",
        },
        {
            "consulta": "#6 Media/mediana de encuentros",
            "resultado": "(59.22327250448558, 36.0)",
            "líneas SQL aprox.": 9,
            "líneas pandas aprox.": 8,
            "tiempo SQL (s)": 0.008418,
            "tiempo pandas (s)": 0.293975,
            "legibilidad": "SQL hace explícita la relación; pandas requiere agrupar y reindexar.",
        },
        {
            "consulta": "#15 Urgencias sin diagnóstico",
            "resultado": "17121",
            "líneas SQL aprox.": 6,
            "líneas pandas aprox.": 7,
            "tiempo SQL (s)": 0.014573,
            "tiempo pandas (s)": 0.439359,
            "legibilidad": "El anti-join es más explícito y compacto en SQL.",
        },
    ]
)
comparison

## Conclusión

Para este volumen (~22,851 pacientes, 1.35 M encounters y hasta 17.3 M observations), SQL/DuckDB es preferible para consultas relacionales, agregaciones y filtros sobre tablas grandes.

pandas sigue siendo útil después de reducir el dataset a un subconjunto analítico manejable para análisis estadístico, ML o visualización.